# 07 — CatBoost vs LightGBM vs blend (sans calibration)

Mêmes features que 06 (te_origin + comportemental + récence + solde). On répond à :
1. Est-ce le **modèle** ? (LightGBM bat-il CatBoost, comme le LGBM de l'ami à 0.3528 ?)
2. Le **blend** des deux dépasse-t-il chaque modèle seul ?

**Aucune calibration** (elle ne sert pas l'AP). On garde la meilleure variante non calibrée.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.blending import rank_average
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6
TE = [C.ORIGIN_ACCT]
WINDOWS = (5, 10, 20)

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)

def feats_train(df, ref):
    X = base_build(df, ref)
    for col in TE:
        X[f"te_{col}"] = oof_target_encode_train(ref, col, C.TARGET)
    return X

def feats_apply(df, ref):
    X = base_build(df, ref)
    for col in TE:
        mp, gm = fit_target_map(ref, col, C.TARGET)
        X[f"te_{col}"] = apply_target_map(df, col, mp, gm)
    return X

def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

def make_lgb():
    import lightgbm as lgb
    return lgb.LGBMClassifier(objective="binary", n_estimators=600, learning_rate=0.05,
                              num_leaves=63, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                              random_state=42, n_jobs=-1, verbose=-1)

## CV : CatBoost, LightGBM, et leur blend (rank-average)

In [ ]:
oof_cat = np.zeros(len(train)); oof_lgb = np.zeros(len(train))
ap_cat, ap_lgb, ap_blend = [], [], []
for tr_idx, va_idx in folds_full:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
    ref = train.iloc[tr_op]
    Xtr = feats_train(train.iloc[tr_op], ref)
    Xva = feats_apply(train.iloc[va_op], ref)
    yt = y_all[tr_op]
    c = make_cat().fit(Xtr, yt); oof_cat[va_op] = c.predict_proba(Xva)[:, 1]
    l = make_lgb().fit(Xtr, yt); oof_lgb[va_op] = l.predict_proba(Xva)[:, 1]
    b = rank_average([oof_cat[va_op], oof_lgb[va_op]])
    ap_cat.append(evaluate_ap(y_all[va_op], oof_cat[va_op]))
    ap_lgb.append(evaluate_ap(y_all[va_op], oof_lgb[va_op]))
    ap_blend.append(evaluate_ap(y_all[va_op], b))
    print(f"fold: cat {ap_cat[-1]:.4f} | lgb {ap_lgb[-1]:.4f} | blend {ap_blend[-1]:.4f}")

def show(name, pf):
    print(f"{name:8s} global {np.mean(pf):.4f} | recent(2) {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f} | LB~ {pf[-1]-0.017:.4f}")
print("\n--- Synthèse (LB~ = last - 0.017) ---")
show("CatBoost", ap_cat); show("LightGBM", ap_lgb); show("Blend", ap_blend)

## Soumission : la meilleure variante (sans calibration)
Ajuste `BEST` selon la synthèse ci-dessus : 'blend', 'lgb' ou 'cat'.

In [ ]:
BEST = "blend"   # <- mettre la variante au meilleur 'last'/'recent'

ref_full = train.iloc[np.where(op03)[0]]
Xf = feats_train(ref_full, ref_full); yf = y_all[op03]
te_op = op03_mask(test).to_numpy()
test_op = test.iloc[np.where(te_op)[0]]
Xte = feats_apply(test_op, ref_full)

cat = make_cat().fit(Xf, yf); pc = cat.predict_proba(Xte)[:, 1]
lgb_ = make_lgb().fit(Xf, yf); pl = lgb_.predict_proba(Xte)[:, 1]
proba = {"cat": pc, "lgb": pl, "blend": rank_average([pc, pl])}[BEST]

full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, f"07_{BEST}")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))